# Matching strategies: de-novo states → ENCODE reference

Strict one-to-one Hungarian objectives for mapping de-novo ChromHMM states to the published ENCODE 15-state reference, with $J$ the Jaccard index and $E$ the emission similarity of a (work, reference) state pair:

1. **Overlap only**: $\max \sum_i |W_i \cap R_{\pi(i)}|$
2. **Emissions only**: $\max \sum_i E(w_i, \pi(w_i))$
3. **Jaccard + emissions**: $\max \sum_i \alpha J + \beta E$
4. **Concave Jaccard + emissions**: $\max \sum_i \alpha \sqrt{J} + \beta E$ — what `match.match_states` optimises, at the weights set in match.py (`JACCARD_WEIGHT`, `EMISSION_WEIGHT`). Its own mapping is reported as **used (match.py)**.
5. **NPMI + emissions**: $\max \sum_i \alpha\,\mathrm{NPMI} + \beta E$

over $(\alpha, \beta) \in \{(1, 0), (0.8, 0.2), (0.5, 0.5), (0.2, 0.8)\}$.

Cohen's $\kappa$ is not among them: for states much smaller than the genome $\kappa \approx$ Dice $= 2J/(1+J)$, monotone in $J$, so it reproduces the Jaccard mappings.

Datasets in the config `outliers` list are excluded.

In [ ]:
import io
import os
import sys
from contextlib import redirect_stderr

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon

for path in ["scripts/rules", "scripts/analysis"]:
    full_path = os.path.abspath(path)
    if full_path not in sys.path:
        sys.path.insert(0, full_path)

import analyze
import match
import utils
from display import show_table

# The project's plot style (fonts, PDF output) from utils, set once for every plot below.
utils.set_style()

with open("config_encode.yaml") as f:
    config = yaml.safe_load(f)

workdir = os.path.expanduser(config.get("workdir", "~/data/2026_segmentations/encode"))
OUTLIERS = set(config.get("outliers", []))
print(f"Working directory: {workdir}")
print(f"Excluded outliers: {sorted(OUTLIERS)}")

OUT = "out/match"
os.makedirs(OUT, exist_ok=True)

## 1. Evaluation

Per dataset and strategy:
- **BP accuracy**: genome share where the mapped work state equals the reference state. In-sample — the overlap objective maximises it by construction, so it bounds the others rather than ranking them.
- **Balanced accuracy**: unweighted mean recall over reference states, $\frac{1}{K} \sum_k |W_{\pi^{-1}(k)} \cap R_k| / |R_k|$.
- **Regulatory / background accuracy**: bp accuracy over reference states typed by `utils` — regulatory = promoter, enhancer and bivalent states; background = `is_noqh` (Quies, Het, ZNF/Rpts, ReprPC).
- **Emission similarity**: mean $E$ over matched pairs.
- **Functional F1, recall, precision**, against annotations not used in matching: active open states vs ATAC-seq, core Tss states vs expressed and RefSeq TSS, Tx states vs expressed gene bodies. Annotation intervals are merged first, as in `analyze.py`. Each benchmark is averaged only over the datasets that have it; `mean_f1` is their mean, defined only for datasets that have all four.

In [ ]:
WEIGHTS = [(1.0, 0.0), (0.8, 0.2), (0.5, 0.5), (0.2, 0.8)]
FAMILIES = ["jaccard", "convex_jaccard", "npmi"]
FUNCTIONAL = {
    # benchmark: (state family of the mapped reference label, annotation)
    "active_atac": utils.is_active_open,
    "tss_exptss": utils.is_promoter_core,
    "tss_refseq": utils.is_promoter_core,
    "tx": utils.is_tx,
}


def is_regulatory(name):
    return utils.is_promoter(name) or utils.is_enhancer(name) or utils.is_biv(name)


def merged(segs):
    # One unnamed interval per covered stretch, so overlapping features are not counted twice.
    by_chrom = {}
    for row in segs:
        by_chrom.setdefault(row[0], []).append([row[1], row[2]])
    return [(chrom, s, e, "feature") for chrom in sorted(by_chrom)
            for s, e in analyze.merge_intervals(by_chrom[chrom])]


def npmi_matrix(overlap, wl, rl, w_states, r_states, genome_len):
    mat = np.full((len(w_states), len(r_states)), -1.0)
    for i, w in enumerate(w_states):
        pw = wl[w] / genome_len
        for j, r in enumerate(r_states):
            pr, pwr = rl[r] / genome_len, overlap.get((w, r), 0) / genome_len
            if 0 < pwr < 1:
                mat[i, j] = np.log(pwr / (pw * pr)) / -np.log(pwr)
    return mat


def hungarian(mat, w_states, r_states):
    rows, cols = linear_sum_assignment(mat, maximize=True)
    return {w_states[i]: r_states[j] for i, j in zip(rows, cols)}


def evaluate_matching_strategies(config, workdir):
    # Run every matching strategy on every non-outlier ENCODE dataset.
    gtf_path = os.path.join(workdir, "gencode.v46.basic.annotation.gtf.gz")
    refseq_tss_segs = merged(match.load_bed(
        os.path.join(workdir, "ChromHMM", "COORDS", "hg38", "RefSeqTSS2kb.hg38.bed.gz")))

    rows, matching_states_records = [], []
    for ds, ds_cfg in config["datasets"].items():
        if ds in OUTLIERS:
            continue
        cell = ds_cfg["cell"]
        ref_acc = ds_cfg.get("ref_chromhmm")
        if not ref_acc:
            continue

        ref_segs_path = os.path.join(workdir, ds, f"{ref_acc}_chromhmm.bed")
        if not os.path.exists(ref_segs_path):
            ref_segs_path += ".gz"
        default_dir = os.path.join(workdir, ds, "chromhmm_default_result")
        work_segs_path = os.path.join(default_dir, f"{cell}_15_dense.bed")
        if not os.path.exists(work_segs_path):
            work_segs_path = os.path.join(default_dir, f"{cell}_15_segments.bed")
        if not os.path.exists(ref_segs_path) or not os.path.exists(work_segs_path):
            continue

        work_segs = match.load_bed(work_segs_path)
        ref_segs = match.load_bed(ref_segs_path)
        wl = match.state_lengths(work_segs)
        rl = match.state_lengths(ref_segs)
        overlap = match.pair_overlap(ref_segs, work_segs)
        w_states = sorted(wl, key=match._natural_sort_key)
        r_states = sorted(rl, key=match._natural_sort_key)
        w_idx = {s: i for i, s in enumerate(w_states)}
        r_idx = {s: i for i, s in enumerate(r_states)}
        genome_len = sum(rl.values()) or 1

        regulatory = [r for r in r_states if is_regulatory(r)]
        background = [r for r in r_states if utils.is_noqh(r)]
        regulatory_bp = sum(rl[r] for r in regulatory)
        background_bp = sum(rl[r] for r in background)

        # Functional annotations: {benchmark: (per-work-state overlap, annotation bp)}
        annotations = {"tss_refseq": refseq_tss_segs}
        atac_acc = ds_cfg.get("atac")
        atac_path = os.path.join(workdir, ds, f"atac_{atac_acc}.bed.gz") if atac_acc else None
        if atac_path and os.path.exists(atac_path):
            annotations["active_atac"] = merged(match.load_bed(atac_path))
        rnaseq_acc = ds_cfg.get("rnaseq")
        rnaseq_path = os.path.join(workdir, ds, f"rnaseq_{rnaseq_acc}.tsv") if rnaseq_acc else None
        if rnaseq_path and os.path.exists(rnaseq_path):
            exp_ann = dict(analyze.make_expressed_annotations(rnaseq_path, gtf_path))
            if exp_ann.get("ExpressedTSS2kb"):
                annotations["tss_exptss"] = merged(exp_ann["ExpressedTSS2kb"])
            if exp_ann.get("ExpressedGeneBodies"):
                annotations["tx"] = merged(exp_ann["ExpressedGeneBodies"])
        features = {}
        for name, segs in annotations.items():
            ov = {w: v for (w, _), v in match.pair_overlap(segs, work_segs).items()}
            features[name] = (ov, sum(e - s for _, s, e, _ in segs))

        work_em = os.path.join(default_dir, f"{cell}_15_dense.bin_emissions.npz")
        if not os.path.exists(work_em):
            work_em = os.path.join(default_dir, f"{cell}_15_dense.bw_emissions.npz")
        ref_em = os.path.join(workdir, ds, f"{ref_acc}_chromhmm.bw_emissions.npz")
        if not os.path.exists(ref_em):
            ref_em = os.path.join(workdir, ds, f"{ref_acc}_chromhmm.bin_emissions.npz")

        # The used mapping, from the project's own matcher; it also writes the matrices to OUT.
        used = match.match_states([ref_segs], [work_segs], matrix_out=os.path.join(OUT, ds),
                                       work_emissions_path=work_em, ref_emissions_path=ref_em)
        # match_states already reported the emission inputs, silence the second load.
        with redirect_stderr(io.StringIO()):
            em_sim = match._load_emission_similarity(work_em, ref_em, w_states, r_states)
        jacc = match.jaccard_matrix(overlap, wl, rl, w_states, r_states)
        utility = (match.JACCARD_WEIGHT * np.sqrt(jacc) + match.EMISSION_WEIGHT * em_sim
                   if em_sim is not None else np.sqrt(jacc))

        for w in w_states:
            r = used.get(w, w)
            i, j = w_idx[w], r_idx.get(r)
            matching_states_records.append({
                "dataset": ds, "cell": cell, "work_state": w, "matched_ref_state": r,
                "jaccard": jacc[i, j] if j is not None else np.nan,
                "emission_sim": em_sim[i, j] if em_sim is not None and j is not None else np.nan,
                "utility": utility[i, j] if j is not None else np.nan,
            })

        base = {
            "jaccard": jacc,
            "convex_jaccard": np.sqrt(jacc),
            "npmi": npmi_matrix(overlap, wl, rl, w_states, r_states, genome_len),
        }
        ov_mat = np.array([[overlap.get((w, r), 0) for r in r_states] for w in w_states], dtype=float)
        mappings = {"overlap": hungarian(ov_mat, w_states, r_states)}
        for family in FAMILIES:
            for a, b in WEIGHTS:
                if b > 0 and em_sim is None:
                    continue
                mat = a * base[family] + (b * em_sim if b > 0 else 0.0)
                mappings[f"{family}_em_{a}_{b}"] = hungarian(mat, w_states, r_states)
        if em_sim is not None:
            mappings["emissions"] = hungarian(em_sim, w_states, r_states)
        mappings["used"] = {w: r for w, r in used.items() if w in w_idx and r in r_idx}

        for s_name, mapping in mappings.items():
            inv_map = {r: w for w, r in mapping.items()}
            hit = {r: overlap.get((inv_map[r], r), 0) if r in inv_map else 0 for r in r_states}
            row = {
                "dataset": ds,
                "strategy": s_name,
                "mapping": ";".join(f"{w}={mapping.get(w, '')}" for w in w_states),
                "accuracy": sum(hit.values()) / genome_len,
                "balanced_accuracy": float(np.mean([hit[r] / rl[r] for r in r_states if rl[r] > 0])),
                "accuracy_regulatory": sum(hit[r] for r in regulatory) / regulatory_bp if regulatory_bp else np.nan,
                "accuracy_background": sum(hit[r] for r in background) / background_bp if background_bp else np.nan,
                "mean_matched_emission_cos": (float(np.mean([em_sim[w_idx[w], r_idx[r]] for w, r in mapping.items()]))
                                              if em_sim is not None else np.nan),
            }
            for name, is_family in FUNCTIONAL.items():
                f1 = prec = rec = np.nan
                if name in features:
                    ov, feature_bp = features[name]
                    family = [w for w, r in mapping.items() if is_family(r)]
                    state_bp = sum(wl[w] for w in family)
                    hit_bp = sum(ov.get(w, 0) for w in family)
                    f1 = 2.0 * hit_bp / (state_bp + feature_bp) if state_bp + feature_bp else 0.0
                    prec = hit_bp / state_bp if state_bp else 0.0
                    rec = hit_bp / feature_bp if feature_bp else 0.0
                row.update({f"f1_{name}": f1, f"precision_{name}": prec, f"recall_{name}": rec})
            # NaN unless all four benchmarks exist, so the mean is one quantity across datasets.
            row["mean_f1"] = float(np.mean([row[f"f1_{name}"] for name in FUNCTIONAL]))
            rows.append(row)
        print(f"Finished {ds}: {len(mappings)} strategies, functional benchmarks {sorted(features)}")

    pd.DataFrame(matching_states_records).to_csv(os.path.join(OUT, "matching_states.tsv"), sep="\t", index=False)
    return pd.DataFrame(rows)

In [ ]:
df_results = evaluate_matching_strategies(config, workdir)
df_results.to_csv(f"{OUT}/matching_strategies_metrics.tsv", sep="\t", index=False)

WEIGHT_TAGS = [f"{a}_{b}" for a, b in WEIGHTS]
FAMILY_LABELS = {"jaccard": "Jaccard", "convex_jaccard": "√Jaccard", "npmi": "NPMI"}
FAMILY_STYLE = {
    # family: (marker, colours from alpha = 1.0 to 0.2)
    "jaccard": ("s", ["#1E8449", "#27AE60", "#52BE80", "#A9DFBF"]),
    "convex_jaccard": ("D", ["#BA4A00", "#E8833A", "#F39C12", "#F8C471"]),
    "npmi": ("v", ["#922B21", "#C0392B", "#E74C3C", "#F1948A"]),
}

strategy_order = ["overlap"]
strategy_labels = {"overlap": "Overlap only"}
strategy_colors = {"overlap": "#4878CF"}
strategy_markers = {"overlap": "X"}
linear_combination_groups = []
for family, (marker, colors) in FAMILY_STYLE.items():
    group = [f"{family}_em_{tag}" for tag in WEIGHT_TAGS]
    for s, (a, b), color in zip(group, WEIGHTS, colors):
        strategy_order.append(s)
        strategy_labels[s] = f"{FAMILY_LABELS[family]} only" if b == 0 else f"{FAMILY_LABELS[family]} + Em ({a}, {b})"
        strategy_colors[s] = color
        strategy_markers[s] = marker
    linear_combination_groups.append((group, colors[1]))
strategy_order += ["emissions", "used"]
strategy_labels.update({"emissions": "Emissions only", "used": "Used"})
strategy_colors.update({"emissions": "#404040", "used": "#000000"})
strategy_markers.update({"emissions": "o", "used": "*"})
strategy_order = [s for s in strategy_order if s in set(df_results["strategy"])]

# The used mapping should be the √Jaccard strategy at match.py's weights.
used_twin = f"convex_jaccard_em_{match.JACCARD_WEIGHT}_{match.EMISSION_WEIGHT}"
maps = df_results.pivot(index="dataset", columns="strategy", values="mapping")
if used_twin in maps:
    print(f"used == {used_twin} on {(maps['used'] == maps[used_twin]).sum()}/{len(maps)} datasets")

SUMMARY_METRICS = ["accuracy", "balanced_accuracy", "accuracy_regulatory", "accuracy_background",
                   "mean_matched_emission_cos"] + [f"f1_{name}" for name in FUNCTIONAL] + ["mean_f1"]
summary = df_results.groupby("strategy")[SUMMARY_METRICS].mean().loc[strategy_order]
summary.insert(0, "label", [strategy_labels[s] for s in summary.index])
summary.round(4).to_csv(f"{OUT}/matching_strategies_summary.tsv", sep="\t")
n_datasets = df_results[df_results["strategy"] == "overlap"][SUMMARY_METRICS].count()
print(f"Evaluated {df_results['dataset'].nunique()} datasets across {len(strategy_order)} strategies.")
print("Datasets per metric:", n_datasets.to_dict())

## 2. Paired differences against overlap matching

Strategies share datasets, so they are compared per dataset against **Overlap only**: mean Δ, the number of datasets whose mapping differs from the overlap mapping at all, how many of those are better / worse, and the Wilcoxon signed-rank p over the datasets where the metric changed. A strategy that changes few mappings can only move the means by a few datasets.

In [ ]:
DELTA_METRICS = ["balanced_accuracy", "accuracy_regulatory", "mean_matched_emission_cos"] + \
                [f"f1_{name}" for name in FUNCTIONAL]
values = {m: df_results.pivot(index="dataset", columns="strategy", values=m) for m in ["accuracy"] + DELTA_METRICS}
delta_rows = []
for s in strategy_order:
    if s == "overlap":
        continue
    changed = maps[s] != maps["overlap"]
    row = {"strategy": s, "label": strategy_labels[s], "mappings_changed": f"{changed.sum()}/{len(changed)}"}
    for m in ["accuracy"] + DELTA_METRICS:
        d = (values[m][s] - values[m]["overlap"]).dropna()
        nz = d[d.abs() > 1e-12]
        row[f"d_{m}"] = d.mean()
        row[f"{m}_better_worse"] = f"{(nz > 0).sum()}/{(nz < 0).sum()} of {len(d)}"
        row[f"p_{m}"] = wilcoxon(nz).pvalue if len(nz) >= 2 else np.nan
    delta_rows.append(row)
df_delta = pd.DataFrame(delta_rows)
df_delta.round(4).to_csv(f"{OUT}/matching_strategies_vs_overlap.tsv", sep="\t", index=False)

## 3. Plots

- **Sensitivity vs coverage** per functional benchmark: recall of the annotation by the mapped state family against its precision; faint points are datasets, markers mean ± SE.
- **Trade-offs**: balanced accuracy against the mean functional F1 (datasets with all four benchmarks only), and emission similarity against regulatory-state accuracy. Lines connect the weight series of one family, from $\alpha = 1$ to $\alpha = 0.2$; in the summary panels the $\alpha = 0.2$ end joins **Emissions only**.
- **Emission similarity** of the matched pairs, and **regulatory vs background accuracy**, per strategy.

In [ ]:
def scatter_strategies(ax, df, x_col, y_col, scale=1.0, show_points=True):
    # Draw every strategy as points + mean ± SE (or the mean alone); return {strategy: (x, y)}.
    sub_df = df.dropna(subset=[x_col, y_col])
    means = {}
    for strat in strategy_order:
        s_data = sub_df[sub_df["strategy"] == strat]
        if s_data.empty:
            continue
        x_vals, y_vals = s_data[x_col] * scale, s_data[y_col] * scale
        means[strat] = (x_vals.mean(), y_vals.mean())
        marker, color = strategy_markers[strat], strategy_colors[strat]
        if show_points:
            ax.scatter(x_vals, y_vals, color=color, marker=marker, alpha=0.3, s=25, edgecolors="none")
            ax.errorbar(*means[strat], xerr=x_vals.sem() if len(x_vals) > 1 else 0,
                        yerr=y_vals.sem() if len(y_vals) > 1 else 0, fmt=marker, color=color,
                        label=strategy_labels[strat], markersize=7.5, markeredgecolor="white",
                        markeredgewidth=1.2, zorder=5)
        else:
            ax.scatter(*means[strat], color=color, marker=marker, label=strategy_labels[strat],
                       s=90 if strat == "used" else 70, edgecolor="white", linewidth=1.2, zorder=5)
    for group, color in linear_combination_groups:
        pts = [means[s] for s in group if s in means]
        if len(pts) > 1:
            ax.plot(*zip(*pts), color=color, linestyle="--", linewidth=1.5, alpha=0.8, zorder=4)
    if not show_points and "emissions" in means:
        for group, _ in linear_combination_groups:
            if group[-1] in means:
                ax.plot(*zip(means[group[-1]], means["emissions"]), color="#B0B0B0",
                        linestyle="--", linewidth=1.2, alpha=0.8, zorder=3)
    return means


def strategy_scatter(df, x_col, y_col, title, xlabel, ylabel, path, scale=1.0, show_points=True):
    fig, ax = plt.subplots(figsize=(6.5, 5))
    scatter_strategies(ax, df, x_col, y_col, scale=scale, show_points=show_points)
    n = df.dropna(subset=[x_col, y_col])["dataset"].nunique()
    ax.set_title(f"{title} (n={n})", **utils.TITLE_STYLE)
    ax.set_xlabel(xlabel, fontsize=utils.AXIS_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=utils.AXIS_FONTSIZE)
    ax.tick_params(labelsize=utils.TICK_FONTSIZE)
    ax.grid(alpha=0.3)
    ax.legend(title="Strategy", fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
    sns.despine(ax=ax)
    utils.save_fig(fig, path)


VALIDATION_PLOTS = [
    ("active_atac", "Active chromatin vs ATAC-seq", "ATAC-seq peak bp covered (%)", "Active state bp in peaks (%)"),
    ("tss_exptss", "Tss states vs expressed TSS", "Expressed TSS bp covered (%)", "Tss state bp at expressed TSS (%)"),
    ("tss_refseq", "Tss states vs RefSeq TSS", "RefSeq TSS bp covered (%)", "Tss state bp at RefSeq TSS (%)"),
    ("tx", "Tx states vs expressed gene bodies", "Expressed gene body bp covered (%)", "Tx state bp in expressed genes (%)"),
]
for name, title, xlabel, ylabel in VALIDATION_PLOTS:
    strategy_scatter(df_results, f"recall_{name}", f"precision_{name}", title, xlabel, ylabel,
                     f"{OUT}/summary_2way_{name}.pdf", scale=100.0)

TRADEOFF_PLOTS = [
    ("balanced_accuracy", "mean_f1", "Balanced accuracy vs functional F1",
     "Balanced accuracy (mean recall over reference states)", "Mean functional F1", "balanced_accuracy_vs_f1"),
    ("mean_matched_emission_cos", "accuracy_regulatory", "Emission similarity vs regulatory accuracy",
     "Mean matched emission similarity", "Regulatory state bp accuracy", "cosine_vs_regulatory"),
]
for x_col, y_col, title, xlabel, ylabel, key in TRADEOFF_PLOTS:
    strategy_scatter(df_results, x_col, y_col, title, xlabel, ylabel, f"{OUT}/matching_tradeoff_{key}.pdf")
    strategy_scatter(df_results, x_col, y_col, f"{title}, means", xlabel, ylabel,
                     f"{OUT}/matching_tradeoff_{key}_summary.pdf", show_points=False)

fig, ax = plt.subplots(figsize=(14, 5.5))
sns.boxplot(data=df_results, x="strategy", y="mean_matched_emission_cos", order=strategy_order,
            hue="strategy", palette=strategy_colors, legend=False, ax=ax, width=0.45, boxprops=dict(alpha=0.7))
utils.strip_points(ax, data=df_results, x="strategy", y="mean_matched_emission_cos", order=strategy_order,
                   dodge=False, size=utils.STRIP_SIZE)
ax.set_xticks(range(len(strategy_order)))
ax.set_xticklabels([strategy_labels[s] for s in strategy_order], rotation=45, ha="right", fontsize=utils.TICK_FONTSIZE)
ax.tick_params(axis="y", labelsize=utils.TICK_FONTSIZE)
ax.set_ylabel("Mean matched emission similarity", fontsize=utils.AXIS_FONTSIZE)
ax.set_xlabel("")
ax.set_title(f"Emission similarity of matched states (n={df_results['dataset'].nunique()})", **utils.TITLE_STYLE)
ax.grid(axis="y", alpha=0.3)
sns.despine(ax=ax)
utils.save_fig(fig, f"{OUT}/matching_semantic.pdf")

CATEGORY_LABELS = {"accuracy_regulatory": "Regulatory (promoter, enhancer, bivalent)",
                   "accuracy_background": "Background (Quies, Het, ZNF/Rpts, ReprPC)"}
df_reg_bg = df_results.melt(id_vars=["strategy", "dataset"], value_vars=list(CATEGORY_LABELS),
                            var_name="category", value_name="acc_value")
df_reg_bg["category_label"] = df_reg_bg["category"].map(CATEGORY_LABELS)
utils.bar_plot(df_reg_bg, x="strategy", y="acc_value", order=strategy_order,
               hue="category_label", hue_order=list(CATEGORY_LABELS.values()),
               palette=dict(zip(CATEGORY_LABELS.values(), ["#3366CC", "#999999"])),
               figsize=(16, 5.5), title=f"Regulatory vs background bp accuracy (n={df_results['dataset'].nunique()})",
               ylabel="BP accuracy", xticklabels=[strategy_labels[s] for s in strategy_order],
               hatch=None, legend=True, legend_title="Reference states", points=dict(size=utils.STRIP_SIZE, alpha=0.5),
               path=f"{OUT}/matching_regulatory_vs_background.pdf")

## 4. Results

In [ ]:
show_table(f"{OUT}/matching_strategies_summary.tsv",
           caption="Mean per strategy. Functional F1 columns average over the datasets with that annotation; mean_f1 over those with all four.")
show_table(f"{OUT}/matching_strategies_vs_overlap.tsv",
           caption="Paired differences against overlap matching (better/worse counted over datasets where the metric changed).")

## 5. Conclusions

Read off the two tables above; the numbers are not repeated here because they move with match.py's weights and the dataset list.

- **BP accuracy is in-sample.** Overlap matching maximises it by construction; only the functional benchmarks are evidence a strategy maps states better.
- **Count the changed mappings before the means.** Most strategies coincide with the overlap mapping on most datasets, so a mean Δ is carried by the few datasets where the bijection actually differs; the better/worse counts and p values say whether it is more than that.
- **Emissions-only matching** gives up the most bp accuracy, mostly on the background states, for its functional gains.